## 1. Estimate a scalar derivative

Estimate the derivative of a scalar polynomial using a forward difference. The Python list `coefficients` stores terms in ascending power order:

$$f(x) = \sum_{k=0}^{n-1} c_k x^k$$

Here $n$ is the coefficient count, $k$ is the power, $c_k$ is its coefficient, and $x$ is the evaluation point. For the supplied positive step $h$, the estimated slope is:

$$\frac{f(x+h) - f(x)}{h}$$

Return a three-item tuple of Python floats in this order: the polynomial value at $x$, its value at $x + h$, and the forward-difference slope.

In [1]:
import numpy as np

def finite_difference_derivative(coefficients: list, x: float, h: float) -> tuple[float, float, float]:
    """
    Returns the value at x, the value at x plus h, and the estimated slope.
    """
    out_x = 0
    out_xh = 0
    for i in range(len(coefficients)):
        out_x += coefficients[i] * np.pow(x,i)
        out_xh += coefficients[i] * np.pow(x+h,i)

    est_slope = (out_xh - out_x) / h

    return tuple([float(out_x),float(out_xh),float(est_slope)])

## 2. Measure Scalar Expression Partials
Estimate the partial derivatives of the scalar expression

The partial-derivative symbol measures sensitivity to one input with the others fixed. Apply the same definition to b and c. Return a four-item tuple of Python floats: the original expression value, then its numerical partial derivatives for a, b, and c.Estimate the partial derivatives of the scalar expression

$$d(a,b,c) = a b + c$$
where, a,b,c are scalars inputs and d is the expression. Use the supplied positive step h and forward differences, such as
$$ \frac{\partial d}{\partial a} \approx \frac{d(a+h,b,c) - d(a,b,c)}{h}$$

The Forward Difference Method: <br>
Instead of using abstract calculus rules to find the exact mathematical formula for the slope, computers often use numerical approximation. The simplest way to do this is the forward difference method.You take your current position, take a tiny microscopic step forward (by adding a small number $h$), and see how much your elevation changed.For your specific expression $d(a,b,c) = ab + c$, here is how you calculate the pieces required for your final output:Original Expression Value:
Calculate the baseline output using the original inputs: $val = a \cdot b + c$.Partial Derivative of $a$:
Add $h$ only to $a$. Keep $b$ and $c$ the same.$$\frac{d(a+h, b, c) - d(a, b, c)}{h}$$Partial Derivative of $b$:
Add $h$ only to $b$. Keep $a$ and $c$ the same.$$\frac{d(a, b+h, c) - d(a, b, c)}{h}$$Partial Derivative of $c$:
Add $h$ only to $c$. Keep $a$ and $b$ the same.$$\frac{d(a, b, c+h) - d(a, b, c)}{h}$$By returning these four calculated numbers in a tuple (val, partial_a, partial_b, partial_c), you give the exact point on the surface and the approximated slope in all three directional dimensions.

In [3]:
def scalar_expression_partials(a: float, b: float, c: float, h: float) -> tuple[float, float, float, float]:
    """
    Returns the expression value and numerical partials for a, b, and c.
    """
    def calc(x,y,z):
        return float(x*y +z)
    d = calc(a,b,c)
    # here the notation i am using is dd/da as d_a and so on
    d_a = (calc(a+h,b,c) - d) / h
    d_b = (calc(a,b+h,c) - d) / h
    d_c = (calc(a,b,c+h) - d) / h

    return tuple([d,d_a,d_b,d_c])

## 3. Check a Product-Chain Gradient

For $L = (ab + c)\,f$, compare the analytic gradient (derived with the chain rule) against a numerical gradient (computed with forward differences) and report the maximum absolute difference.

#### A. Analytic Gradients (The Exact Math)

Analytic gradients use calculus and the chain rule to find the exact rate of change. We work backward from the output $L$ to the inputs $a, b, c, f$.

First, calculate the "local" gradients at the final step $L = e \cdot f$:

$$
\frac{\partial L}{\partial f} = e, \qquad \frac{\partial L}{\partial e} = f
$$

If you change $f$, $L$ changes by a factor of $e$.

Next, use the chain rule to pass that gradient further back to $a, b,$ and $c$ through the intermediate variable $e = ab + c$:

$$
\frac{\partial L}{\partial a} = \frac{\partial L}{\partial e} \cdot \frac{\partial e}{\partial a} = f \cdot b
$$

$$
\frac{\partial L}{\partial b} = \frac{\partial L}{\partial e} \cdot \frac{\partial e}{\partial b} = f \cdot a
$$

$$
\frac{\partial L}{\partial c} = \frac{\partial L}{\partial e} \cdot \frac{\partial e}{\partial c} = f \cdot 1 = f
$$

Your analytic list will be `[f * b, f * a, f, e]`.

#### B. Numerical Gradients (The Approximation)

Numerical gradients ignore the chain rule entirely. Instead, they treat the entire equation as a "black box" function:

$$
F(a, b, c, f) = (ab + c)\,f
$$

To find the gradient for a specific input, you just add a tiny step $h$ to that input while keeping all others frozen, and measure how much the output changed:

$$
\frac{\partial L}{\partial a} \approx \frac{F(a+h, b, c, f) - F(a, b, c, f)}{h}
$$

$$
\frac{\partial L}{\partial b} \approx \frac{F(a, b+h, c, f) - F(a, b, c, f)}{h}
$$

$$
\frac{\partial L}{\partial c} \approx \frac{F(a, b, c+h, f) - F(a, b, c, f)}{h}
$$

$$
\frac{\partial L}{\partial f} \approx \frac{F(a, b, c, f+h) - F(a, b, c, f)}{h}
$$

#### C. The Comparison

Because the numerical gradient is just an approximation, it won't perfectly match the exact analytic math — but for a very small $h$ (like `1e-5`), it should be incredibly close.

To prove they match, calculate the absolute difference between the analytic and numerical gradient for each of the four inputs, and find the maximum difference among them:

$$
\max\left(\left|\frac{\partial L}{\partial a}_{analytic} - \frac{\partial L}{\partial a}_{numeric}\right|, \ldots\right)
$$

Your final output is a single tuple containing all these pieces:

```python
(L, analytic_list, numerical_list, max_difference)
```

In [1]:
def gradient_check_product_chain(
    a: float,
    b: float,
    c: float,
    f: float,
    h: float,
) -> tuple[float, list, list, float]:
    """
    Returns loss, analytic gradients, numerical gradients, and maximum error.
    """
    e = float(a*b +c)
    # for numerical gradients (approximation)
    def calc(a,b,c,f):
        return float(((a*b)+c)*f)
    d = calc(a,b,c,f)
    d_a = (calc(a+h,b,c,f) - d) / h
    d_b = (calc(a,b+h,c,f) - d) / h
    d_c = (calc(a,b,c+h,f) - d) / h    
    d_f = (calc(a,b,c,f+h) - d) / h    
    num_gradients = [float(d_a),float(d_b),float(d_c),float(d_f)]

    # for analytical gradients, based on the partial derivative
    analytic_gradients = [float(f*b),float(f*a),float(f),e]

    # Loss
    L = float(e*f)

    #max error
    max_diff = max(abs(a_grad - n_grad) for a_grad, n_grad in zip(analytic_gradients, num_gradients))
    return tuple([L,analytic_gradients,num_gradients,max_diff])
    
    

## 4. Gradient-Descent Step


Apply one gradient-descent update to the Python lists `values` and `gradients`:

$$
\theta_i' = \theta_i - \eta\, g_i
$$

Here $i$ indexes coordinates, $\theta_i$ is the original value, $g_i$ is its gradient, $\eta$ is `learning_rate`, and $\theta_i'$ is the updated value.

The predicted first-order objective change is:

$$
\Delta L_{pred} = \sum_i g_i \left(\theta_i' - \theta_i\right)
$$

Here $\Delta L_{pred}$ is a **local prediction**, not a measured change in loss.

### Return

A two-item tuple:

```python
(updated_values, predicted_change)
```

- `updated_values` — a fresh list of updated Python floats
- `predicted_change` — the predicted change as a Python float

In [2]:
def gradient_descent_step(values: list, gradients: list, learning_rate: float) -> tuple[list, float]:
    """
    Returns a fresh list of updated values and the predicted objective change.
    """
    values,gradients = np.array(values,dtype=float), np.array(gradients,dtype=float)
    old_values = values.copy()
    values -= learning_rate * gradients
    pred_obj = np.sum(gradients * (values - old_values))
    return tuple([values.tolist(),float(pred_obj)])